# PharmaShield GCC — Layer 4: Ranking & Dual-Source Order Split
Plan §4.2, §4.4, §5.2 steps 7–8. Run notebooks 01 and 02 first (they fill `data/processed/suppliers_matrix.csv`). Team decisions L1–L5 are in `docs/DECISIONS.md`.

GenAI disclosure (plan §6.1): code generated with Claude and reviewed by the team.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'layer4_decision').exists())
sys.path.insert(0, str(ROOT / 'layer4_decision'))
from decision import engine
from decision.weights import MODES
m, tiers, sku = engine.load_inputs()

## Step 7 — weights per mode and AHP consistency

In [2]:
res = {mode: engine.run(m, tiers, sku, mode, lam=1.0, entropy_share=0.0) for mode in MODES}
print('AHP consistency ratio:', res['Normal']['ahp_consistency']['CR'])
pd.concat({mode: r['weights']['Used for ranking'] for mode, r in res.items()}, axis=1)

Running HiGHS 1.16.0 (git hash: f3f113d): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
AHP consistency ratio: 0.0


,Normal,Crisis,Cost Focus
criterion,,,
C1_landed_cost,0.10,0.00,0.20
C2_delivery_risk,0.30,0.30,0.30
C3_price_integrity,0.05,0.05,0.05
C4_compliance_risk,0.30,0.30,0.30
C5_route_exposure,0.25,0.35,0.15


### Why the entropy share is 0 by default (decision L5): entropy weights are shown for information

In [3]:
res['Normal']['entropy_table']

,criterion,varies_across_makers,entropy,divergence
0,C1_landed_cost,True,0.9603,0.0397
1,C2_delivery_risk,True,0.4364,0.5636
2,C3_price_integrity,False,NaN,0.0000
3,C4_compliance_risk,True,0.9469,0.0531
4,C5_route_exposure,True,0.4324,0.5676


## TOPSIS ranking (S = closeness to the ideal supplier; higher is better)

In [4]:
pd.DataFrame({mode: r['ranking']['S'] for mode, r in res.items()}).sort_values('Normal', ascending=False).round(3)

,Normal,Crisis,Cost Focus
manufacturer_id,,,
BPI LABS,0.736,0.700,0.787
GLAND PHARMA,0.736,0.700,0.787
APOTEX,0.694,0.649,0.758
NATCO PHARMA USA,0.694,0.649,0.758
TEYRO LABS,0.694,0.649,0.758
AVYXA PHARMA,0.654,0.858,0.493
TEVA PARENTERAL MEDICINES,0.632,0.591,0.708
HOSPIRA,0.586,0.518,0.681
EUGIA US,0.536,0.456,0.643


## Step 8 — order split for 10,000 vials (PuLP)

In [5]:
for mode, r in res.items():
    a = r['allocation']
    print(mode, r['diagnostics']['solver_status'], '| spot buy', r['diagnostics']['spot_buy_vials'])
    display(a[a['selected']][['manufacturer_id', 'factory_group', 'S', 'vials', 'share_pct', 'role']])

Normal Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.736406,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.694264,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.632286,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.585968,1000.0,10.0,Additional source


Crisis Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.700163,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.649258,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.590596,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.518483,1000.0,10.0,Additional source


Cost Focus Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.786642,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.757728,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.707875,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.681129,1000.0,10.0,Additional source


## Sensitivity: the risk dial λ

In [6]:
pd.DataFrame({lam: engine.run(m, tiers, sku, 'Normal', lam=lam)['allocation'].set_index('manufacturer_id')['vials']
              for lam in (0.0, 0.5, 1.0, 2.0)}).loc[lambda d: d.sum(axis=1) > 0]

,0.0,0.5,1.0,2.0
manufacturer_id,,,,
ACCORD HEALTHCARE,3000.0,0.0,0.0,0.0
APOTEX,3000.0,3000.0,3000.0,3000.0
BPI LABS,3000.0,3000.0,3000.0,3000.0
EUGIA US,1000.0,0.0,0.0,0.0
HOSPIRA,0.0,1000.0,1000.0,1000.0
TEVA PARENTERAL MEDICINES,0.0,3000.0,3000.0,3000.0
